### KPIs: avg idle time

In [ ]:
from clickhouse_driver import Client


client = Client(
        host='localhost',
        port=int(9000),
        user='click',
        password='click',
        database='pl',
    )

with open("../sql/idle_time.sql", "r") as f:
    query = f.read()

df = client.query_dataframe(query)
# df

# debug
# 
# df[df.fleet_id != 7].describe()
# df[df.trip_hours < 0].describe()


In [ ]:
# df.hist()

In [ ]:
# add fleet-driver idle time
SHIFT = 8

# df['idle_time'] = 8 - df.trip_hours
df['idle_time'] = (SHIFT - df['trip_hours']).clip(lower=0)

# limit trip hours to reasonable 15 hours time
df.trip_hours = df.trip_hours.apply(lambda x: x if x <= 15 else 15)

# df

In [ ]:
# get fleet idle time
df_fleet = df.groupby(by=['fleet_id']).mean('idle_time').reset_index()


In [ ]:
# df_fleet

In [ ]:
import plotly.express as px

# sort by idle time
df_plot = df_fleet.sort_values("idle_time", ascending=False).copy()

# fleet_id to str to keep only existing fleets on x
df_plot['fleet_id'] = df_plot['fleet_id'].astype(str)

df_plot['idle_percent'] = (df_plot['idle_time'] / SHIFT * 100).round(1)


fig = px.bar(
    df_plot,
    x="fleet_id",
    y="idle_time",
    color="fleet_id",               
    text="idle_percent",    
    title="Fleet Idle Time (Daily)",
    labels={
        "fleet_id": "Fleet ID",
        "idle_time": "Idle Time",
    }
)

# txt improvement
fig.update_traces(
    texttemplate="%{text}%",
    textposition='outside',
    hovertemplate='<b>Fleet %{x}</b><br>Idle Time: %{text}%'
)

fig.update_layout(
    title_font_size=24,
    yaxis=dict(range=[0, 10]),
    legend_title_text="Fleet",
    legend=dict(font=dict(size=12)),
    template="plotly_white"
)

fig.show()
